# Build Flow for Heatmap Model (KV260)

In this notebook, we will use the FINN compiler to generate an FPGA accelerator for the `heatmap_model.onnx`. 
This notebook is based on the "Advanced Builder Settings" tutorial but adapted for the specific requirements of the heatmap model and the **Kria KV260** hardware.

## Outline
---------------

1. [Setup and Model Preparation](#setup)
2. [Shape Inference Repair](#shape_fix)
3. [Reference Data Generation](#ref_data)
4. [Builder Flow Configuration](#builder)
5. [Resource Estimates & Inspection](#inspection)

In [1]:
import os
import shutil
import numpy as np
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
from finn.util.visualization import showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup
from qonnx.core.onnx_exec import execute_onnx

# Setup paths
# Ensure 'heatmap_model.onnx' is in the same directory as this notebook
model_filename = "heatmap_model.onnx"
build_dir = os.environ['FINN_ROOT'] + "/notebooks/heatmap_build_kv260"

# Create build directory
if not os.path.exists(build_dir):
    os.makedirs(build_dir)
    print(f"Created build directory: {build_dir}")

# Define path for the working copy of the model
clean_model_path = build_dir + "/heatmap_model_clean.onnx"

# Copy the original model to the build directory to preserve the source
if os.path.exists(model_filename):
    shutil.copy(model_filename, clean_model_path)
    print(f"Model copied to: {clean_model_path}")
else:
    raise FileNotFoundError(f"Could not find {model_filename}. Please upload it to the notebook directory.")

Model copied to: /home/kokinama/finn/notebooks/heatmap_build_kv260/heatmap_model_clean.onnx


## Shape Inference Repair <a id='shape_fix'></a>

The provided model contains static shape information (e.g., hardcoded dimensions) that conflicts with the calculated shapes during inference. This often leads to "Dimension Mismatch" errors in the compiler.

To fix this, we apply a "radical" cleanup step before starting the builder flow. This removes all stored `value_info` metadata and forces ONNX to recalculate shapes fresh from the input.

In [2]:
print(f"Repairing shapes in: {clean_model_path}")

# 1. Load Model
model = ModelWrapper(clean_model_path)

# 2. Radical Cure: Delete ALL stored shape information
# This forces ONNX to recalculate everything fresh from the input during the next cleanup/inference.
while len(model.graph.value_info) > 0:
    model.graph.value_info.pop()

# 3. Save and run cleanup
model.save(clean_model_path)
cleanup(clean_model_path, out_file=clean_model_path)

print("✅ Model shapes have been reset and recalculated.")

Repairing shapes in: /home/kokinama/finn/notebooks/heatmap_build_kv260/heatmap_model_clean.onnx
✅ Model shapes have been reset and recalculated.


Let's visualize the cleaned model to ensure it looks correct before proceeding.

In [3]:
showInNetron(clean_model_path)

Serving '/home/kokinama/finn/notebooks/heatmap_build_kv260/heatmap_model_clean.onnx' at http://0.0.0.0:8081


## Reference Data Generation <a id='ref_data'></a>

Since the model uses custom QONNX quantization nodes (e.g., `Quant`), standard `onnxruntime` cannot be used for verification as it does not support these operations.

Instead, we use the **QONNX execution engine** (`execute_onnx`) to generate reference `input.npy` and `expected_output.npy` files. This Python-based simulation correctly handles the custom nodes.

In [4]:
# ==============================================================================
# CELL 4: Radikale Shape-Entfernung (Die "Nuklear-Option")
# ==============================================================================
from qonnx.core.modelwrapper import ModelWrapper

# Pfad definieren
clean_model_path = build_dir + "/heatmap_model_clean.onnx"

print(f"🔨 Erzwinge Shape-Bereinigung für: {clean_model_path}")

# 1. Modell laden
model = ModelWrapper(clean_model_path)

# 2. Alle Zwischenspeicher-Shapes löschen (Hier steht oft die falsche "45")
if len(model.graph.value_info) > 0:
    del model.graph.value_info[:]

# 3. Alle Output-Shapes löschen (Hier kann der Fehler auch liegen)
for out in model.graph.output:
    if out.type.tensor_type.HasField("shape"):
        out.type.tensor_type.ClearField("shape")

# 4. Speichern OHNE 'cleanup' oder 'infer_shapes'
# Wir speichern das Modell jetzt "dumm" (ohne Größenangaben).
# Wir führen KEINE Überprüfung durch, damit es nicht sofort wieder abstürzt.
# Der FINN-Compiler wird im späteren Schritt (step_streamline) die
# korrekten Werte (80) automatisch berechnen.
model.save(clean_model_path)

print("✅ Modell wurde 'nackt' (ohne Shapes) gespeichert.")
print("👉 Bitte führen Sie jetzt direkt die Builder-Zelle (Cell 10/12) aus.")

Starte manuelle Reparatur für: /home/kokinama/finn/notebooks/heatmap_build_kv260/heatmap_model_clean.onnx
   (Diagnose) Anzahl gespeicherter Shapes vor Löschen: 175
   (Diagnose) Anzahl gespeicherter Shapes nach Löschen: 0
   -> Zwischenspeicher erfolgreich geleert.
✅ Modell wurde ohne Shapes gespeichert in: /home/kokinama/finn/notebooks/heatmap_build_kv260/heatmap_model_clean.onnx
Bitte führen Sie nun den Rest des Notebooks (ab Cell 9/10) aus.


## Builder Flow Configuration <a id='builder'></a>

We now set up the FINN builder. 

**Hardware Target:** We are targeting the **Xilinx Kria KV260** Vision AI Starter Kit. 
The FPGA part for the K26 SOM is `xck26-sfvc784-2LV-c`.

We will use the `estimate_only_dataflow_steps` flow first to generate resource estimates without running full synthesis (which saves time).

In [5]:
output_dir = build_dir + "/output_estimates"

# Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")

# Define the steps (using standard estimate flow)
# You can inject custom steps here if pre/post processing is needed later
build_steps = build_cfg.estimate_only_dataflow_steps

cfg_estimates = build.DataflowBuildConfig(
    output_dir          = output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 10000,
    synth_clk_period_ns = 10.0, 
    fpga_part           = "xck26-sfvc784-2LV-c", # Target for Kria KV260 (K26 SOM)
    steps               = build_steps,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
    # Optional: Enable verification if desired, using the generated npy files
    # verify_steps=[
    #     build_cfg.VerificationStepType.TIDY_UP_PYTHON,
    #     build_cfg.VerificationStepType.STREAMLINED_PYTHON
    # ]
)

Previous run results deleted!


Run the builder flow. This performs tidy-up, streamlining, conversion to HW layers, and resource estimation.

In [ ]:
%%time
build.build_dataflow_cfg(clean_model_path, cfg_estimates)

Running step: step_qonnx_to_finn [1/10]


Building dataflow accelerator from /home/kokinama/finn/notebooks/heatmap_build_kv260/heatmap_model_clean.onnx
Intermediate outputs will be generated in /home/kokinama/finn/finn_dev_kokinama
Final outputs will be generated in /home/kokinama/finn/notebooks/heatmap_build_kv260/output_estimates
Build log is at /home/kokinama/finn/notebooks/heatmap_build_kv260/output_estimates/build_dataflow.log
Running step: step_qonnx_to_finn [1/10]


Running step: step_tidy_up [2/10]
Running step: step_streamline [3/10]


Running step: step_tidy_up [2/10]
Running step: step_streamline [3/10]
> /usr/local/lib/python3.10/dist-packages/onnx/shape_inference.py(40)infer_shapes()
     38     if isinstance(model, (ModelProto, bytes)):
     39         model_str = model if isinstance(model, bytes) else model.SerializeToString()
---> 40         inferred_model_str = C.infer_shapes(
     41             model_str, check_type, strict_mode, data_prop
     42         )



Traceback (most recent call last):
  File "/home/kokinama/finn/src/finn/builder/build_dataflow.py", line 158, in build_dataflow_cfg
    model = transform_step(model, cfg)
  File "/home/kokinama/finn/src/finn/builder/build_dataflow_steps.py", line 323, in step_streamline
    model = model.transform(Streamline())
  File "/home/kokinama/finn/deps/qonnx/src/qonnx/core/modelwrapper.py", line 140, in transform
    (transformed_model, model_was_changed) = transformation.apply(transformed_model)
  File "/home/kokinama/finn/src/finn/transformation/streamline/__init__.py", line 98, in apply
    model = model.transform(trn)
  File "/home/kokinama/finn/deps/qonnx/src/qonnx/core/modelwrapper.py", line 140, in transform
    (transformed_model, model_was_changed) = transformation.apply(transformed_model)
  File "/home/kokinama/finn/src/finn/transformation/streamline/reorder.py", line 670, in apply
    model = model.transform(InferShapes())
  File "/home/kokinama/finn/deps/qonnx/src/qonnx/core/modelwr

## Inspection <a id='inspection'></a>

The output directory contains intermediate models and reports. Let's inspect the sequence of generated intermediate ONNX files to understand how the compiler transformed the graph.

In [ ]:
!ls -t -r {output_dir}/intermediate_models

We can visualize the model after `step_convert_to_hw` to see which layers were successfully converted to hardware (HLS/RTL) layers (look for `fpgadataflow` nodes).

In [ ]:
showInNetron(output_dir + "/intermediate_models/step_convert_to_hw.onnx")

Finally, let's examine the estimated resource usage (LUTs, BRAMs, DSPs) for the Kria KV260.

In [ ]:
import json

report_path = output_dir + "/report/estimate_layer_resources.json"
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    print("Total Resource Estimation:")
    print(json.dumps(report["total"], indent=2))
else:
    print("Report not found. Please check the build logs for errors.")